# （発展・任意）ステップ6：機械が決めた基準と、自分が決めた基準を比べる

これは探究講座の **任意** のステップです。時間が余った班・興味のある人向け。

ステップ4 で、あなたは重みを決めて「有人基地に向く場所／向かない場所」を選びました。
その「向き・不向き」を、**機械学習に当てさせる** とどうなるでしょう。

- **やること**：「日照率」と「緯度」の2つから、その場所が有人基地に向くかを当てるモデルをつくる
- **モデルは5種類**：ロジスティック回帰／決定木／k近傍／ニューラルネット／ランダムフォレスト
- **仕組みは覚えなくてよい**。`method=` を取り替えて、境界線の形・当たりやすさ・読みやすさを比べる

## 進め方
1. まずワークシートに **予想** を書く（どれが一番当たる？　どれが一番「読める」？）
2. セルを実行して、5つのモデルの結果を見る
3. 5つを順位づけして、理由を書く

In [ ]:
from moonkit import *
from moonkit_ml import train, METHODS

# ステップ4と同じ「有人基地スコア」を計算し、その上位25%を「有人基地むき」とする
極 = dist_to_permanent_shadow(south_pole(load('極域日照')))
極['スコア'] = site_score(極, {
    'average_illumination_percent': ('高い', 1),
    'km_to_shadow':                 ('低い', 1),
    'permanent_shadow_fraction':    ('低い', 1),
}, top=None)['スコア']
極['有人基地むき'] = (極['スコア'] >= 極['スコア'].quantile(0.75)).astype(int)

ラベル = 極['有人基地むき']
print('南極の地点数：', len(極), '／ そのうち「有人基地むき」：', int(ラベル.sum()),
      f'（{ラベル.mean()*100:.0f}%）')

my_threshold = 5     # ★ここを変える：ステップ3であなたが決めた「日照率のしきい値」[%]（参考線）

---
## 5つのモデルを全部ためす

下のセルを実行すると、5つのモデルそれぞれについて：
- **色つきの地図**：そのモデルが「有人基地むき」と判定する範囲（オレンジ）と、そうでない範囲（青）
- **点**：テストデータの正解（オレンジ＝むき、青＝むかない）
- **赤い点線**：ステップ3で決めた日照率のしきい値（参考）
- **正解率**：テストデータで何％当たったか
- **ルールを言葉で説明できる？**

**注意**：モデルが見ているのは「日照率」と「緯度」の2つだけ。でも「有人基地むき」は
**氷までの距離（`km_to_shadow`）にも左右される**。つまり、描いてある2つの軸だけでは
正解が完全には決まらない。

In [ ]:
結果 = {}
for m in METHODS:
    r = train(極, features=['average_illumination_percent', 'lat'], label=ラベル,
              method=m, my_threshold=my_threshold)
    結果[m] = r['テスト正解率']
    print()

print('=== まとめ ===')
for m, acc in sorted(結果.items(), key=lambda kv: -kv[1]):
    print(f'  {m:12s} テスト正解率 {acc:.3f}')

**ワークシートに記録**：5つのモデルの正解率。1位と最下位の差は何ポイント？
境界線の形は5つでどう違った？（まっすぐ／階段状／ぐにゃぐにゃ／なめらか／ブロック）

**気づいたこと**：正解率が 100% にならないのはなぜ？（ヒント：モデルは氷までの距離を見ていない）

---
## 「複雑さ」を上げると何が起きる？（過学習の実験）

`複雑さ='高い'` にすると、k近傍は「となりの1点だけ」を見るようになり、
ニューラルネットは大きくなります。境界線がどうなるか見てみましょう。

In [ ]:
train(極, ['average_illumination_percent', 'lat'], ラベル,
      method='k近傍', 複雑さ='高い', my_threshold=my_threshold)

In [ ]:
train(極, ['average_illumination_percent', 'lat'], ラベル,
      method='ランダムフォレスト', 複雑さ='高い', my_threshold=my_threshold)

**気づいたこと**：境界線が「飛び地」だらけになっていませんか？
練習データの正解率は上がるのに、テストデータの正解率は上がらない（むしろ下がる）ことがあります。
これを **過学習（かがくしゅう）** といいます。「練習問題を丸暗記したけれど、本番の問題は解けない」状態です。

弱い構造（描いてある2軸の中にある、ちょっとしたかたより）を、柔軟なモデルは拾おうとします。
それが「本物の規則」なのか「たまたまのノイズ」なのかは、練習データとテストデータの
正解率の差で見分けます。

---
## 考える（ワークシートに書く）

1. **正解率で順位をつける**と？　1位と最下位の差は何ポイント？　「けっこう横並び」だと思う？
2. **「ルールを言葉で説明できるか」で順位をつける**と？（1と順位は同じ？　ちがう？）
3. 正解率が 100% にならない＝「日照率と緯度」だけでは決まらない、ということ。
   では、あと何が分かれば正解率は上がる？
4. ステップ4 では、あなたは3つの条件に重みをつけて場所を選びました。
   その「あなたのルール」と、機械が学習したルールは、似ている？　ちがう？
5. **宇宙飛行士が住む場所を決めるとき**、正解率が少し高いけれど「なぜそう判定したか説明できない」モデルと、
   正解率が少し低いけれど「if〜then のルールが読める」モデル、どちらを使う？　なぜ？